In [3]:
import geopandas as gpd
import pandas as pd
import sqlite3
import os


In [4]:
os.chdir("/Users/terezasaskova/Library/Mobile Documents/com~apple~CloudDocs/3-30-300-index")
buildings_routed = gpd.read_file('data/buildings_routed.gpkg', driver='GPKG', layer='buildings')
parks = gpd.read_file('data/buildings_routed.gpkg', driver='GPKG', layer='parks')

/opt/anaconda3/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(
/opt/anaconda3/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(


In [7]:
# save to sqlite and geojson
conn = sqlite3.connect("data/buildings_routed.db")
buildings_routed.drop(columns="geometry").to_sql("buildings", conn, if_exists="replace", index=False)
conn.close()
buildings_routed.to_file("data/buildings_routed.geojson", driver="GeoJSON")

In [8]:
# ── STATS ──────────────────────────────────────────────────────────────────────

total = len(buildings_routed)
within = buildings_routed["within_300m"].sum()
outside = total - within

print("300m PARK ACCESS INDICATOR — BOLOGNA")

print(f"\n── ADDRESS POINTS ──")
print(f"  Total civici:                {total:>8,}")
print(f"  Within 300m walk of park:    {within:>8,}  ({within/total*100:.1f}%)")
print(f"  Outside 300m:                {outside:>8,}  ({outside/total*100:.1f}%)")
print(f"  No park reachable (NaN):     {buildings_routed['park_id'].isna().sum():>8,}")

print(f"\n── PARKS ──")
print(f"  Total parks (all layers):    {len(parks):>8,}")
print(f"  by layer type:")
# load full parks for layer breakdown
parks_full = (
    pd.concat([
        gpd.read_file("data/greenery__all_size.gpkg", layer=l)
        for l in ["ar_vrd_all", "pe_unis_all", "osm_all_public"]
    ], ignore_index=True)
)
print(parks_full["layer_type"].value_counts().to_string())


300m PARK ACCESS INDICATOR — BOLOGNA

── ADDRESS POINTS ──
  Total civici:                  77,630
  Within 300m walk of park:      53,442  (68.8%)
  Outside 300m:                  24,188  (31.2%)
  No park reachable (NaN):       24,188

── PARKS ──
  Total parks (all layers):         698
  by layer type:
layer_type
ar_vrd_all         3814
park_garden_all    1030
pe_unis_all         542


In [9]:

print(f"\n── AR_VRD FILTERING ──")
ar_vrd_all   = (parks_full["layer_type"] == "ar_vrd_all").sum()
ar_vrd_kept  = (parks["layer_type"] == "ar_vrd_all").sum() if "layer_type" in parks.columns else "n/a"
ar_vrd_removed = ar_vrd_all - ar_vrd_kept if isinstance(ar_vrd_kept, int) else "n/a"
print(f"  ar_vrd before filter:        {ar_vrd_all:>8,}")
print(f"  ar_vrd after filter (<=6):   {ar_vrd_kept:>8}")
print(f"  ar_vrd removed:              {ar_vrd_removed:>8}")
print(f"  Filter logic: removed ar_vrd polygons intersecting >6 building")
print(f"  footprints — proxy for private residential block interiors")



── AR_VRD FILTERING ──
  ar_vrd before filter:           3,814
  ar_vrd after filter (<=6):        481
  ar_vrd removed:                   n/a
  Filter logic: removed ar_vrd polygons intersecting >6 building
  footprints — proxy for private residential block interiors


In [10]:

print(f"\n── PARK SIZE DISTRIBUTION (reachable parks) ──")
if "park_area_ha" in buildings_routed.columns:
    sizes = buildings_routed.dropna(subset=["park_area_ha"])["park_area_ha"]
    print(f"  Mean park area:   {sizes.mean():.2f} ha")
    print(f"  Median park area: {sizes.median():.2f} ha")
    print(f"  Min:              {sizes.min():.2f} ha")
    print(f"  Max:              {sizes.max():.2f} ha")

print(f"\n── PARK TYPE BREAKDOWN (matched to buildings) ──")
if "park_type" in buildings_routed.columns:
    print(buildings_routed[buildings_routed["within_300m"]]
          ["park_type"].value_counts().to_string())

print("=" * 55)


── PARK SIZE DISTRIBUTION (reachable parks) ──
  Mean park area:   1.01 ha
  Median park area: 0.24 ha
  Min:              0.00 ha
  Max:              45.60 ha

── PARK TYPE BREAKDOWN (matched to buildings) ──
park_type
ar_vrd_all         39431
park_garden_all    10048
pe_unis_all         3963


In [7]:
CRS_WGS = "EPSG:4326"
CRS_METRIC  = "EPSG:32632"
import warnings
warnings.filterwarnings("ignore")
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
from tqdm import tqdm
import requests
from shapely.geometry import Point
from shapely.strtree import STRtree
 
# 3. parks - read ALL layers together from the start
parks = (
    pd.concat([
        gpd.read_file("data/greenery__all_size.gpkg", layer=l)
        for l in ["ar_vrd_all", "pe_unis_all", "osm_all_public"]
    ], ignore_index=True)
    .to_crs(CRS_METRIC)
    .reset_index(drop=True)
    .assign(park_id=lambda df: df.index)
)

print(f"{len(parks):,} parks total before filtering")
print(parks["layer_type"].value_counts())

# filter ar_vrd using building footprints
building_footprints = gpd.read_file("data/bologna_buildings_final.gpkg").to_crs(CRS_METRIC)
building_footprints = building_footprints.loc[
    building_footprints.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()

footprint_tree = STRtree(building_footprints.geometry.values)

def buildings_inside(park_geom):
    candidates = footprint_tree.query(park_geom, predicate="intersects")
    return len(candidates)

# compute only for ar_vrd, fill 0 for others
ar_vrd_mask = parks["layer_type"] == "ar_vrd_all"
parks.loc[ar_vrd_mask, "n_buildings_inside"] = (
    parks.loc[ar_vrd_mask, "geometry"].apply(buildings_inside)
)
parks["n_buildings_inside"] = parks["n_buildings_inside"].fillna(0)

# apply filter only to ar_vrd
parks = parks[
    ~ar_vrd_mask | (parks["n_buildings_inside"] <= 6)
].reset_index(drop=True)

# reassign park_id after filtering
parks["park_id"] = parks.index

print(f"{len(parks):,} parks after filtering")
print(parks["layer_type"].value_counts())

# ── AR_VRD THRESHOLD JUSTIFICATION ──
ar_vrd_full = (gpd.read_file("data/greenery__all_size.gpkg", layer="ar_vrd_all")
               .to_crs(CRS_METRIC))
ar_vrd_full["n_buildings_inside"] = ar_vrd_full.geometry.apply(buildings_inside)

print("Distribution of n_buildings_inside for ar_vrd:")
print(ar_vrd_full["n_buildings_inside"].describe().round(2))

# show how many parks are removed at different thresholds
for thresh in [2, 4, 6, 8, 10, 15]:
    kept = (ar_vrd_full["n_buildings_inside"] <= thresh).sum()
    removed = len(ar_vrd_full) - kept
    print(f"  threshold <= {thresh:2d}: kept {kept:4d}, removed {removed:4d} ({removed/len(ar_vrd_full)*100:.1f}%)")

5,386 parks total before filtering
layer_type
ar_vrd_all         3814
park_garden_all    1030
pe_unis_all         542
Name: count, dtype: int64
3,807 parks after filtering
layer_type
ar_vrd_all         2235
park_garden_all    1030
pe_unis_all         542
Name: count, dtype: int64
Distribution of n_buildings_inside for ar_vrd:
count    3814.00
mean       10.50
std        16.95
min         0.00
25%         1.00
50%         4.00
75%        14.00
max       201.00
Name: n_buildings_inside, dtype: float64
  threshold <=  2: kept 1599, removed 2215 (58.1%)
  threshold <=  4: kept 1958, removed 1856 (48.7%)
  threshold <=  6: kept 2235, removed 1579 (41.4%)
  threshold <=  8: kept 2453, removed 1361 (35.7%)
  threshold <= 10: kept 2616, removed 1198 (31.4%)
  threshold <= 15: kept 2974, removed  840 (22.0%)


In [10]:
!pip install --upgrade tqdm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.66.1
    Uninstalling tqdm-4.66.1:
      Successfully uninstalled tqdm-4.66.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.6.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.5.1 which is incompatible.
crewai 1.7.1 requires click~=8.1.7, but you have click 8.3.1 which is incompatible.
crewai 1.7.1 requires openai~=1.83.0, but you have openai 2.16.0 which is incompatible.
crewai 1.7.1 requires pydantic~=2.11.9, but you have pydantic 2.12.5 which is incompatible.
crewai 1.7.1 requires python-dotenv~=1.1.1, but you have python-dotenv 1.0.0 which is incompatible.
crewai 1.7.1 requires tokenizers~=0.20.3, but you have tokenizers 0.22.1 which is incompatib

In [ ]:
GH_URL = "http://localhost:8989/route"

def graphhopper(orig_lat, orig_lon, dest_lat, dest_lon):
    params = {
        "point": [f"{orig_lat},{orig_lon}", f"{dest_lat},{dest_lon}"],
        "profile": "foot",
        "calc_points": "false",
        "instructions": "false",
    }
    try:
        r = requests.get(GH_URL, params=params, timeout=5)
        r.raise_for_status()
        return r.json()["paths"][0]["distance"]
    except Exception:
        return None

# only compute for buildings within 300m that have an entrance
mask = buildings_routed["within_300m"] & buildings_routed["entrance_lat"].notna()
subset = buildings_routed[mask].copy()

# get building coords
subset_wgs = subset.copy()
subset_wgs["orig_lat"] = [p.y for p in subset.geometry.to_crs(CRS_WGS)]
subset_wgs["orig_lon"] = [p.x for p in subset.geometry.to_crs(CRS_WGS)]

from tqdm.notebook import tqdm

results = []
for _, row in tqdm(subset_wgs.iterrows(), total=len(subset_wgs)):
    results.append(graphhopper(
        row["orig_lat"], row["orig_lon"],
        row["entrance_lat"], row["entrance_lon"]
    ))

subset_wgs["walk_dist_m"] = results



  0%|          | 0/53442 [00:00<?, ?it/s]

  0%|          | 0/53442 [14:47<?, ?it/s]


In [19]:
print("buildings_routed columns:", buildings_routed.columns.tolist())
print("subset_wgs columns:", subset_wgs.columns.tolist())
print("subset_wgs index sample:", subset_wgs.index[:5].tolist())
print("buildings_routed index sample:", buildings_routed.index[:5].tolist())
print("walk_dist_m in subset_wgs:", "walk_dist_m" in subset_wgs.columns)
print("walk_dist_m null count:", subset_wgs["walk_dist_m"].isna().sum() if "walk_dist_m" in subset_wgs.columns else "MISSING")

buildings_routed columns: ['codvia', 'street_number', 'subciv', 'codarco', 'codfab', 'partkey', 'ncivsub', 'civkey', 'cap', 'denominazi', 'quartiere', 'zona', 'zona_nome', 'id_civ', 'bacino_scu', 'bacino_sc0', 'bacino_sc1', 'data_istit', 'data_varia', 'geo_point_2d', 'x_utm_etrs', 'y_utm_etrs', 'sez_1981_i', 'sez_1991_i', 'sez_2001_i', 'sez_2011_i', 'cod_loc_is', 'loc_istat', 'centro_sto', 'area_stati', 'cod_areast', 'sez_istat', 'codice_ann', 'codice_ans', 'zona_pross', 'address', 'building_id', 'park_id', 'entrance_lat', 'entrance_lon', 'within_300m', 'park_area_ha', 'park_source', 'park_type', 'park_name', 'geometry', 'walk_dist_m_x', 'walk_dist_m_y']
subset_wgs columns: ['codvia', 'street_number', 'subciv', 'codarco', 'codfab', 'partkey', 'ncivsub', 'civkey', 'cap', 'denominazi', 'quartiere', 'zona', 'zona_nome', 'id_civ', 'bacino_scu', 'bacino_sc0', 'bacino_sc1', 'data_istit', 'data_varia', 'geo_point_2d', 'x_utm_etrs', 'y_utm_etrs', 'sez_1981_i', 'sez_1991_i', 'sez_2001_i', 'sez_

In [21]:
# Drop the duplicate columns from previous merge attempts
buildings_routed = buildings_routed.drop(columns=["walk_dist_m_x", "walk_dist_m_y"], errors="ignore")

# Write walk distances directly back using the original index
buildings_routed["walk_dist_m"] = None
buildings_routed.loc[subset_wgs.index, "walk_dist_m"] = walk_distances

# Restore as GeoDataFrame (in case previous steps degraded it)
import geopandas as gpd
buildings_routed = gpd.GeoDataFrame(buildings_routed, geometry="geometry", crs=buildings_routed.crs)

# Reproject and export
gdf_out = buildings_routed.to_crs("EPSG:4326")
gdf_out["longitude"] = gdf_out.geometry.x
gdf_out["latitude"]  = gdf_out.geometry.y

final_cols = {
    "ncivsub":      "street_number",
    "address":      "address",
    "longitude":    "longitude",
    "latitude":     "latitude",
    "quartiere":    "neighbourhood",
    "park_id":      "nearest_park_id",
    "park_type":    "park_type",
    "park_source":  "park_source",
    "park_area_ha": "park_area_ha",
    "entrance_lat": "entrance_lat",
    "entrance_lon": "entrance_lon",
    "walk_dist_m":  "walk_dist_m",
    "within_300m":  "within_300m",
}
import geopandas as gpd

# Include geometry in the selection, then rename
keep_cols = list(final_cols.keys()) + ["geometry"]
buildings_out = gpd.GeoDataFrame(
    gdf_out[keep_cols].rename(columns=final_cols),
    geometry="geometry",
    crs=gdf_out.crs
)

buildings_out.to_file("data/buildings_300m_indicator.geojson", driver="GeoJSON")
buildings_out.drop(columns="geometry").to_csv("data/buildings_300m_indicator.csv", index=False)

import sqlite3
conn = sqlite3.connect("data/buildings_300m_indicator.db")
buildings_out.drop(columns="geometry").to_sql("buildings", conn, if_exists="replace", index=False)
conn.close()

print("Saved: GeoJSON, CSV, SQLite")
print(buildings_out.head())

Saved: GeoJSON, CSV, SQLite
  street_number                             address  longitude   latitude  \
0          27/1        VIA MARCO EMILIO LEPIDO 27/1  11.278541  44.516819   
1          46/2  VIA GIOVANNI BATTISTA GANDINO 46/2  11.355349  44.478504   
2           9/2              VIA GUIDO ZUCCHINI 9/2  11.351607  44.501726   
3             2               VIA RAGAZZI DEL '99 2  11.312626  44.502516   
4           2/2               VIA DELLA FILANDA 2/2  11.297137  44.493548   

           neighbourhood  nearest_park_id    park_type park_source  \
0  Borgo Panigale - Reno            755.0   ar_vrd_all        DBSN   
1          Santo Stefano           2451.0  pe_unis_all        DBSN   
2          Santo Stefano              NaN          NaN         NaN   
3      Porto - Saragozza            505.0   ar_vrd_all        DBSN   
4  Borgo Panigale - Reno              NaN          NaN         NaN   

   park_area_ha  entrance_lat  entrance_lon walk_dist_m  within_300m  \
0      0.117755 